# Stability Analysis

This section provides the necessary calculations for the stability analysis in chapter 6.1.1. More specifically, it calculates the raw maximum coefficient of variation for each experiment and the final value after the removal of a outlier explained in chapter 6.1.1.

## Imports

In [1]:
import numpy as np
import pandas as pd

## Data Preparation

In [2]:
baseline = pd.read_csv("raw_data/0.csv")
A1 = pd.read_csv("raw_data/A1.csv")
A2 = pd.read_csv("raw_data/A2.csv")
B = pd.read_csv("raw_data/B.csv")
C = pd.read_csv("raw_data/C.csv")

CV calculation before exclusion of A2_8 outlier configuration.

In [3]:
def calc_cv(data):
    stability_results = data.groupby(["id", "stage"])[["agent_0_prices", "agent_1_prices"]].agg(["mean", "std", "count"])

    # Flatten the MultiIndex columns
    stability_results.columns = [f"{col}_{stat}" for col, stat in stability_results.columns]

    # Standard error
    stability_results["agent_0_prices_se"] = stability_results["agent_0_prices_std"] / np.sqrt(stability_results["agent_0_prices_count"])
    stability_results["agent_1_prices_se"] = stability_results["agent_1_prices_std"] / np.sqrt(stability_results["agent_1_prices_count"])

    # Coefficient of variation
    stability_results["agent_0_prices_cv"] = (
        stability_results["agent_0_prices_std"] / stability_results["agent_0_prices_mean"]
    )
    stability_results["agent_1_prices_cv"] = (
        stability_results["agent_1_prices_std"] / stability_results["agent_1_prices_mean"]
    )

    return stability_results

In [4]:
config_batches = {
    "baseline": baseline,
    "A1": A1,
    "A2": A2,
    "B": B,
    "C": C
}
summary = {}
for name, batch in config_batches.items():
    stability_results = calc_cv(batch)
    summary[name] = stability_results
    print(f"Experiment {name} stability results: {round(stability_results["agent_0_prices_cv"].max(), 4)} for agent 0, {round(stability_results["agent_1_prices_cv"].max(),4)} for agent 1")

Experiment baseline stability results: 0.0194 for agent 0, 0.0192 for agent 1
Experiment A1 stability results: 0.0311 for agent 0, 0.0305 for agent 1
Experiment A2 stability results: 0.3056 for agent 0, 0.1991 for agent 1
Experiment B stability results: 0.0216 for agent 0, 0.0224 for agent 1
Experiment C stability results: 0.0132 for agent 0, 0.012 for agent 1


Observation: Experiment A2 has very high CV compared to other Experiments. Lets examine CV of each configuration.

In [5]:
print(summary["A2"][["agent_0_prices_cv", "agent_1_prices_cv"]])

            agent_0_prices_cv  agent_1_prices_cv
id   stage                                      
A2_0 0               0.008088           0.007465
     1               0.007901           0.007274
     2               0.008055           0.007277
A2_1 0               0.005884           0.004890
     1               0.011887           0.011206
     2               0.014642           0.014116
A2_2 0               0.007086           0.005802
     1               0.015956           0.014857
     2               0.020507           0.020049
A2_3 0               0.013641           0.009850
     1               0.008976           0.007594
     2               0.003826           0.004506
A2_4 0               0.031427           0.032781
     1               0.014413           0.012842
     2               0.015053           0.013496
A2_5 0                    NaN                NaN
     1               0.069803           0.085928
     2               0.043897           0.053369
A2_6 0              

Observation: Unusually high CV for experiment 8. We investigate outliers:

In [6]:
rows = []

for col in ["agent_0_prices", "agent_1_prices"]:
    tmp = A2.copy()
    grouped = tmp.groupby(["id", "stage"])[col]
    median = grouped.transform("median")
    mad = grouped.transform(lambda x: np.median(np.abs(x - np.median(x))))

    # avoid division by zero
    robust_z = 0.6745 * (tmp[col] - median) / mad.replace(0, np.nan)
    tmp["Outlier column"] = col
    tmp["Z-Score"] = robust_z

    out = tmp[tmp["Z-Score"].abs() > 7]
    rows.append(out)

price_outliers = pd.concat(rows, ignore_index=True)
price_outliers[
    ["id", "seed", "stage", "Outlier column", "agent_0_prices", "agent_1_prices", "Z-Score", "rel. loss0", "rel. loss1"]
].sort_values(["id", "seed", "stage"])

,id,seed,stage,Outlier column,agent_0_prices,agent_1_prices,Z-Score,rel. loss0,rel. loss1
0,A2_8,8,1,agent_0_prices,0.075646,0.291828,-15.537654,1.0,1.0
1,A2_8,8,1,agent_1_prices,0.075646,0.291828,-9.021527,1.0,1.0
2,A2_8,8,2,agent_1_prices,1.054937,0.687055,-7.557816,1.0,1.0


Observation: A1_8 has large outliers with a absolute z-score of over 15 for seed 8. Furthermore, its relative loss is also abnormaly higher than the one of its peers. Hence we remove it as a learning anomaly that has not converged. 

In [7]:
config_batches["A2"] = A2[~((A2["id"] == "A1_8") & (A2["seed"] == 8))]

## Table 6.1

In [8]:
for name, batch in config_batches.items():
    stability_results = calc_cv(batch)
    print(f"Experiment {name} stability results: {round(stability_results["agent_0_prices_cv"].max(), 4)} for agent 0, {round(stability_results["agent_1_prices_cv"].max(),4)} for agent 1")

Experiment baseline stability results: 0.0194 for agent 0, 0.0192 for agent 1
Experiment A1 stability results: 0.0311 for agent 0, 0.0305 for agent 1
Experiment A2 stability results: 0.3056 for agent 0, 0.1991 for agent 1
Experiment B stability results: 0.0216 for agent 0, 0.0224 for agent 1
Experiment C stability results: 0.0132 for agent 0, 0.012 for agent 1


Results look a lot better now. CV now low accross all experiments and acceptable for A1_7 with $\le 0.2$.